# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 53), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.87 MiB | 10.37 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/FlyRank-Internship


In [2]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

## 1. Method choice and why

**Method: Logistic Regression only.**

My Week 4 baseline confirmed a real, strong signal (CTR consistently drops as position
gets worse, with a clean monotonic relationship across large samples) but used it as a
fixed rule — a hand-picked score formula, not a trained model. This week's job is to
check whether a real model, trained on the same underlying signals, can predict CTR
underperformance more reliably than my rule did — and specifically whether a *simple*
model can do this, before reaching for anything more complex.

I'm choosing Logistic Regression specifically (not Random Forest or Gradient Boosting)
because the assignment explicitly warns against rewarding complexity for its own sake.
My Week 4 signal check already showed the core relationship (CTR vs. position) is fairly
clean and monotonic — exactly the kind of pattern a linear model can capture well. If
Logistic Regression already performs solidly, there's no honest reason to reach for a
more complex, less interpretable method just to get a marginal lift. This also keeps
every coefficient interpretable, which matters for a lane about explainable,
decision-support scoring, not black-box prediction.

**Target label:** 1 if a page's actual CTR falls meaningfully below its position
bucket's expected CTR (using the same expected-CTR-by-bucket reference established in
Week 4's Signal Check 2), 0 otherwise. This turns my Week 4 rule's continuous score into
a proper binary target that both the rule and a real model can be evaluated against,
on the same basis.

In [3]:
# Build the modeling dataset: same March 2026 slice, same features used in Week 4

model_data = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    ),
    joined AS (
        SELECT
            m.*,
            c.word_count,
            c.search_volume,
            c.competition,
            c.cpc,
            c.backlinks,
            c.content_type,
            c.main_intent
        FROM march_data m
        JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
            ON m.content_hash_id = c.content_hash_id
    )
    SELECT *,
        clicks_march * 1.0 / NULLIF(impressions_march, 0) AS actual_ctr,
        CASE
            WHEN avg_position_march <= 3 THEN '1. position 1-3'
            WHEN avg_position_march <= 10 THEN '2. position 4-10'
            WHEN avg_position_march <= 20 THEN '3. position 11-20'
            ELSE '4. position 20+'
        END AS position_bucket
    FROM joined
    WHERE impressions_march >= 500 AND avg_position_march > 0 AND avg_position_march <= 20
""").df()

# Apply the same expected-CTR-by-bucket reference from Week 4
bucket_expected = {
    '1. position 1-3': 0.003765,
    '2. position 4-10': 0.003208,
    '3. position 11-20': 0.002625,
}
model_data['expected_ctr'] = model_data['position_bucket'].map(bucket_expected)
model_data['label'] = (model_data['actual_ctr'] < model_data['expected_ctr'] * 0.7).astype(int)

print("Shape:", model_data.shape)
print("Label distribution:\n", model_data['label'].value_counts())
model_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (50764, 16)
Label distribution:
 label
1    26345
0    24419
Name: count, dtype: int64


,content_hash_id,client_hash_id,impressions_march,clicks_march,avg_position_march,word_count,search_volume,competition,cpc,backlinks,content_type,main_intent,actual_ctr,position_bucket,expected_ctr,label
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,2123,20,0.00,0.00,<NA>,keyword article,informational,0.001073,2. position 4-10,0.003208,1
1,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,2546,10,0.00,0.00,<NA>,keyword article,informational,0.001066,2. position 4-10,0.003208,1
2,content_05434271b257bb68,client_73cda7b4e4f265ea,1421.0,6.0,6.320337,<NA>,10,0.00,0.00,<NA>,keyword article,commercial,0.004222,2. position 4-10,0.003208,0
3,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.950311,2809,30,0.00,0.00,<NA>,keyword article,informational,0.003803,2. position 4-10,0.003208,0
4,content_aafb2ab7e5fc80d0,client_73cda7b4e4f265ea,7709.0,20.0,5.258331,2556,720,0.24,1.05,<NA>,keyword article,commercial,0.002594,2. position 4-10,0.003208,0


## 2. Split design

**Grouped split by client_hash_id.** Content items from the same client likely share
patterns (site structure, content style, template quality) that have nothing to do with
the actual signal I'm trying to model. If the same client's pages appear in both train
and test, the model could partly "memorize" that client rather than learning a
generalizable pattern — making my test score look better than it would on a genuinely
new client. A grouped split keeps every client's pages entirely in one side (train or
test), so the test set genuinely represents unseen clients, matching how this model
would actually need to perform in practice — being applied to a new client's content,
not just new pages from an already-seen client.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_data, groups=model_data['client_hash_id']))

train_df = model_data.iloc[train_idx].reset_index(drop=True)
test_df = model_data.iloc[test_idx].reset_index(drop=True)

print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)
print("Train clients:", train_df['client_hash_id'].nunique(), "| Test clients:", test_df['client_hash_id'].nunique())

# Confirm no client overlap between train and test
overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print("Client overlap (should be 0):", len(overlap))

Train shape: (19150, 16) | Test shape: (31614, 16)
Train clients: 24 | Test clients: 11
Client overlap (should be 0): 0


**Note on split proportions:** although `test_size=0.3` was requested, the actual result
is 24 clients/19,150 rows in train vs. 11 clients/31,614 rows in test — test ended up
larger by row count. This is expected behavior for a grouped split on an unbalanced
panel: since clients have very different numbers of pages, the 70/30 target applies
loosely to rows, but the real unit being split is clients, and a few large clients
landing in the test group can shift the row ratio substantially. This doesn't undermine
the split's validity (0 client overlap is what actually matters for honesty), but it's
worth naming rather than assuming the split behaved as a naive 70/30 would.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score
import numpy as np

# Numeric features only for now (Logistic Regression needs clean numeric input)
numeric_features = ['impressions_march', 'avg_position_march', 'word_count',
                     'search_volume', 'competition', 'cpc', 'backlinks']

train_clean = train_df.dropna(subset=numeric_features + ['label']).copy()
test_clean = test_df.dropna(subset=numeric_features + ['label']).copy()

print("Train after dropping NAs:", train_clean.shape)
print("Test after dropping NAs:", test_clean.shape)

X_train = train_clean[numeric_features]
y_train = train_clean['label']
X_test = test_clean[numeric_features]
y_test = test_clean['label']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

pred_proba = model.predict_proba(X_test_scaled)[:, 1]
pred_label = model.predict(X_test_scaled)

model_auc = roc_auc_score(y_test, pred_proba)
model_precision = precision_score(y_test, pred_label)

print(f"Logistic Regression — ROC AUC: {model_auc:.3f}")
print(f"Logistic Regression — Precision: {model_precision:.3f}")

Train after dropping NAs: (11776, 16)
Test after dropping NAs: (15603, 16)
Logistic Regression — ROC AUC: 0.508
Logistic Regression — Precision: 0.531


In [7]:
# Inspect what the model actually learned
import pandas as pd

coef_df = pd.DataFrame({
    'feature': numeric_features,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df)

# Also check: how much data did we lose to the word_count/backlinks NA drop?
print("\nOriginal train size:", train_df.shape[0], "-> after dropna:", train_clean.shape[0])
print("Original test size:", test_df.shape[0], "-> after dropna:", test_clean.shape[0])

              feature  coefficient
1  avg_position_march     0.281335
0   impressions_march    -0.248729
4         competition    -0.111624
3       search_volume     0.092246
5                 cpc     0.028719
2          word_count     0.018735
6           backlinks    -0.013744

Original train size: 19150 -> after dropna: 11776
Original test size: 31614 -> after dropna: 15603


In [8]:
# Drop backlinks (mostly null, weakest coefficient) instead of dropping rows for it
numeric_features_v2 = ['impressions_march', 'avg_position_march', 'word_count',
                        'search_volume', 'competition', 'cpc']

train_clean_v2 = train_df.dropna(subset=numeric_features_v2 + ['label']).copy()
test_clean_v2 = test_df.dropna(subset=numeric_features_v2 + ['label']).copy()

print("Train after dropping NAs (v2):", train_clean_v2.shape, "| was:", train_clean.shape)
print("Test after dropping NAs (v2):", test_clean_v2.shape, "| was:", test_clean.shape)

X_train_v2 = train_clean_v2[numeric_features_v2]
y_train_v2 = train_clean_v2['label']
X_test_v2 = test_clean_v2[numeric_features_v2]
y_test_v2 = test_clean_v2['label']

scaler_v2 = StandardScaler()
X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2)
X_test_v2_scaled = scaler_v2.transform(X_test_v2)

model_v2 = LogisticRegression(max_iter=1000, random_state=42)
model_v2.fit(X_train_v2_scaled, y_train_v2)

pred_proba_v2 = model_v2.predict_proba(X_test_v2_scaled)[:, 1]
pred_label_v2 = model_v2.predict(X_test_v2_scaled)

model_v2_auc = roc_auc_score(y_test_v2, pred_proba_v2)
model_v2_precision = precision_score(y_test_v2, pred_label_v2)

print(f"\nLogistic Regression v2 (no backlinks) — ROC AUC: {model_v2_auc:.3f}")
print(f"Logistic Regression v2 (no backlinks) — Precision: {model_v2_precision:.3f}")

Train after dropping NAs (v2): (17541, 16) | was: (11776, 16)
Test after dropping NAs (v2): (21663, 16) | was: (15603, 16)

Logistic Regression v2 (no backlinks) — ROC AUC: 0.543
Logistic Regression v2 (no backlinks) — Precision: 0.567


In [9]:
# Baseline: Week 4's rule-based flag (actual_ctr below expected_ctr), evaluated on the same test set
baseline_pred = (test_clean_v2['actual_ctr'] < test_clean_v2['expected_ctr']).astype(int)

baseline_precision = precision_score(y_test_v2, baseline_pred)
baseline_auc = roc_auc_score(y_test_v2, -test_clean_v2['actual_ctr'])  # lower actual_ctr = higher risk score

print(f"Week 4 baseline rule — Precision: {baseline_precision:.3f}")
print(f"Week 4 baseline rule (ranked by actual_ctr) — ROC AUC: {baseline_auc:.3f}")

# Final comparison table
import pandas as pd
comparison = pd.DataFrame({
    'Method': ['Week 4 baseline (rule-based)', 'Week 5 Logistic Regression'],
    'ROC AUC': [baseline_auc, model_v2_auc],
    'Precision': [baseline_precision, model_v2_precision]
})
print("\n", comparison)

Week 4 baseline rule — Precision: 0.802
Week 4 baseline rule (ranked by actual_ctr) — ROC AUC: 0.997

                          Method   ROC AUC  Precision
0  Week 4 baseline (rule-based)  0.996520   0.801675
1    Week 5 Logistic Regression  0.542615   0.567416


**Critical finding: this comparison is invalid, and that is the actual result.**

The numbers above (Week 4 baseline ROC AUC = 0.997, Precision = 0.802 vs. Logistic
Regression ROC AUC = 0.543, Precision = 0.567) look like the baseline dramatically beat
the model — but this comparison is circular, not fair, and reporting it at face value
would be dishonest.

The label was defined as `actual_ctr < expected_ctr * 0.7`. The "baseline" I evaluated
against it was `actual_ctr < expected_ctr` — almost the same condition, just with a
looser threshold. The baseline isn't predicting real-world underperformance from
independent signals; it's nearly reproducing the exact rule the label was built from. A
0.997 AUC here is the same warning sign as the deliberate leakage trap in Week 3
(0.902 honest vs. 1.000 leaked) — an unrealistically perfect score that means the
"prediction" and the "answer" are too close to independent.

**What this actually proves:** my label definition is too tightly coupled to my Week 4
rule to serve as a fair target for testing whether that same rule is a good baseline.
This is a genuine methodology lesson, not a modeling failure — the Logistic Regression's
weak but honest performance (AUC 0.543, a real if modest improvement over chance) is
more trustworthy than the baseline's inflated 0.997, precisely because the model was
learning from independent features (position, impressions, word count, search volume,
competition, CPC) rather than a near-restatement of the label itself.

**What a valid comparison would require:** a label built from a source genuinely
independent of the CTR-vs-expected-CTR rule — for example, a future-window outcome
(did this page's CTR actually improve after a real intervention), or an entirely
different signal like engagement/session behavior. Building that label is future work;
this week's honest conclusion is that my baseline "beat" my model only because the
comparison itself was flawed by construction, not because the rule is genuinely
superior.

## 4. Errors and interpretation

Since Section 3 established that the baseline comparison itself was invalid, this
section focuses on what the Logistic Regression genuinely learned, independent of that
flawed comparison.

**What the model leans on:** `avg_position_march` (coefficient +0.281) and
`impressions_march` (coefficient -0.249) are by far the strongest signals — worse
position and higher impressions both push toward the "underperforming" label. This is
directionally sensible: a page seen many times but ranking well should, intuitively, be
converting better, so a poor CTR at high impressions and good position is a genuine
anomaly worth flagging. The remaining features (competition, search_volume, cpc,
word_count) contribute only weakly.

**Why the AUC is still low despite sensible coefficients:** an AUC of 0.543 means the
model finds a real but weak pattern — position and impressions correlate with the label
in the expected direction, but not strongly enough to reliably separate the two classes.
This matches Week 4's own finding: Signal Check 2 confirmed CTR drops with worse
position, but the relationship, while statistically clean, is not steep enough on its
own to predict a binary "underperforming" outcome with high confidence at the individual
page level.

**What this means for the lane going forward:** position and impressions alone are not
sufficient features for this label. A stronger version of this model would need either
richer features (e.g., actual title/meta text signals, historical CTR trend rather than
a single-month snapshot) or, more importantly, a genuinely independent label not derived
from the same rule being tested — as discussed in Section 3.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.